# Fine-tune EmbeddingGemma — Swiss Legal Retrieval
Dataset: `farwew/swiss-legal` (anchor, positive, negative triplets)

In [ ]:
!pip install -q -U sentence-transformers datasets
!pip install -q git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

In [ ]:
from huggingface_hub import login
login()  # masukkan HF token (butuh akses ke dataset private + model gemma)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model_id = 'google/embeddinggemma-300m'
model = SentenceTransformer(model_id, device=device)

# Gradient checkpointing — hemat ~60% VRAM, sedikit lebih lambat
model[0].auto_model.gradient_checkpointing_enable()
print('Gradient checkpointing enabled')
print('Max seq length:', model.max_seq_length)

In [ ]:
from datasets import load_dataset

ds = load_dataset('farwew/swiss-legal')
print(ds)
print('Columns:', ds['train'].column_names)
print('Train size:', len(ds['train']))

# Tampilkan sample
sample = ds['train'][0]
print('
Sample:')
print('  anchor  :', sample['anchor'][:100])
print('  positive:', sample['positive'][:100])
print('  negative:', sample['negative'][:100])

In [ ]:
# Cek similarity sebelum training
import numpy as np

samples = ds['train'].select(range(100))
a_emb = model.encode(samples['anchor'],   prompt_name='Retrieval-query',    normalize_embeddings=True)
p_emb = model.encode(samples['positive'], prompt_name='Retrieval-document', normalize_embeddings=True)
n_emb = model.encode(samples['negative'], prompt_name='Retrieval-document', normalize_embeddings=True)

pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)

print('=== SEBELUM TRAINING ===')
print(f'Positive > Negative: {(pos_sim > neg_sim).sum()}/100')
print(f'Avg positive sim   : {pos_sim.mean():.3f}')
print(f'Avg negative sim   : {neg_sim.mean():.3f}')

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

train_dataset = ds['train'].select_columns(['anchor', 'positive', 'negative'])
eval_dataset  = ds['eval'].select_columns(['anchor', 'positive', 'negative']) if 'eval' in ds else None

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='finetuned-embeddinggemma-swiss-legal',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    prompts={
        'anchor':   model.prompts['Retrieval-query'],
        'positive': model.prompts['Retrieval-document'],
        'negative': model.prompts['Retrieval-document'],
    },
    logging_steps=50,
    report_to='none',
    save_strategy='epoch',
    save_total_limit=1,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

In [ ]:
# Cek similarity setelah training
a_emb = model.encode(samples['anchor'],   prompt_name='Retrieval-query',    normalize_embeddings=True)
p_emb = model.encode(samples['positive'], prompt_name='Retrieval-document', normalize_embeddings=True)
n_emb = model.encode(samples['negative'], prompt_name='Retrieval-document', normalize_embeddings=True)

pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)

print('=== SETELAH TRAINING ===')
print(f'Positive > Negative: {(pos_sim > neg_sim).sum()}/100')
print(f'Avg positive sim   : {pos_sim.mean():.3f}')
print(f'Avg negative sim   : {neg_sim.mean():.3f}')

In [ ]:
# Simpan model
model.save('finetuned-embeddinggemma-swiss-legal/final')
print('Model saved.')

# Optional: push ke HuggingFace Hub
# model.push_to_hub('farwew/embeddinggemma-swiss-legal')